# BCQ for Offline 2D Navigation

This notebook is the presentation companion for the project. It explains the environment, shows one numerical BCQ critic update, runs one actual neural-network update, and evaluates the saved policy.

Run it from the project root. In Google Colab, upload the project files (including `data/` and `checkpoints/`) first, then run the cells in order.

## 1. Environment and offline-learning setting

The agent moves in a 10 × 10 map from the lower-left start region to the upper-right goal, while avoiding the central obstacle. The state is `[x, y, goal_x - x, goal_y - y]` and the continuous action is `[a_x, a_y] ∈ [-1, 1]^2`.

BCQ is **offline RL**: first we save transitions `(s, a, r, s', done)` from an imperfect behavior policy; after that, training uses only this fixed dataset.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from bcq import BCQ
from config import DATASET_PATH, MODEL_PATH, SEED
from dataset import OfflineDataset
from env import NavigationEnv
from visualize import draw_environment

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Dataset exists:', Path(DATASET_PATH).exists())
print('Checkpoint exists:', Path(MODEL_PATH).exists())

In [ ]:
env = NavigationEnv()
state = env.reset()
print('Initial state:', state)
print('Action dimension:', env.action_dim, ' Action range: [-1, 1]')

fig, ax = plt.subplots(figsize=(6, 6))
draw_environment(ax)
ax.set_title('2D navigation environment')
plt.show()

## 2. Fixed dataset

The behavior policy follows imperfect waypoint routes and adds noise. Thus, the data include good actions but also suboptimal moves and collisions. BCQ must improve the policy without querying the environment for new training samples.

In [ ]:
data = np.load(DATASET_PATH)
for key in data.files:
    print(f'{key:12s}', data[key].shape)
print('Number of transitions:', len(data['states']))
print('One transition:')
print('state     =', data['states'][0])
print('action    =', data['actions'][0])
print('reward    =', data['rewards'][0, 0])
print('next state=', data['next_states'][0])
print('done      =', bool(data['dones'][0, 0]))

## 3. One BCQ value update by hand

For a next state, the VAE generates actions that resemble the dataset. The perturbation actor changes each action only slightly. The two critics score them with:

`Qmix = 0.75 × min(Q1, Q2) + 0.25 × max(Q1, Q2)`

The highest-scoring candidate defines the Bellman target: `y = r + γ(1-done) max(Qmix)`.

In [ ]:
# Didactic numerical example: not a sample from the trained network.
gamma, reward, done = 0.99, 1.0, 0.0
q1_candidates = np.array([8.0, 7.0])
q2_candidates = np.array([6.0, 7.0])
q_mix = 0.75 * np.minimum(q1_candidates, q2_candidates) + 0.25 * np.maximum(q1_candidates, q2_candidates)
best_q = q_mix.max()
target_y = reward + (1 - done) * gamma * best_q

current_q1, current_q2 = 5.0, 4.5
critic_loss = (current_q1 - target_y) ** 2 + (current_q2 - target_y) ** 2

print('Qmix for candidates:', q_mix)       # [6.5, 7.0]
print('Selected next-state Q:', best_q)     # 7.0
print('Bellman target y:', round(target_y, 2))  # 7.93
print('Critic loss:', round(critic_loss, 2))    # 20.35
print('Gradient descent changes critic weights so Q1 and Q2 move toward y.')

## 4. One actual neural-network BCQ update

The code below takes a minibatch from the fixed dataset and performs the three real updates: VAE reconstruction, double-critic Bellman regression, and perturbation-actor optimization. It creates a new in-memory agent, so the saved presentation checkpoint is unchanged.

In [ ]:
dataset = OfflineDataset(DATASET_PATH, device='cpu')
learning_agent = BCQ(state_dim=4, action_dim=2, max_action=1.0, device='cpu')
losses = learning_agent.train(dataset.sample(batch_size=256))
for name, value in losses.items():
    print(f'{name}: {value:.4f}')

## 5. Evaluate the saved policy

For each state, the trained BCQ policy samples 100 dataset-like candidate actions, applies a small perturbation, evaluates them with the critic, and executes the best one.

In [ ]:
from evaluate import evaluate_bcq

agent = BCQ(state_dim=4, action_dim=2, max_action=1.0, device='cpu')
agent.load(MODEL_PATH)
metrics, trajectory = evaluate_bcq(agent, episodes=100, seed=SEED + 4000)
print(metrics)

fig, ax = plt.subplots(figsize=(6, 6))
draw_environment(ax)
ax.plot(trajectory[:, 0], trajectory[:, 1], '-o', ms=3, lw=2, label='BCQ')
ax.legend()
ax.set_title('One trained BCQ rollout')
plt.show()

## Conclusion

BCQ achieves high reward by choosing from actions supported by the offline dataset rather than freely extrapolating to arbitrary continuous actions. In the fixed-task benchmark, the saved model reaches the goal reliably while avoiding collisions.